# Chunk 35 — Classification Metrics

## Pre-registered definitions

**LABEL_MIN:** `y = 1[min(true S11) < -10 dB]`. **Score:** `-min(predicted S11)`, so higher means more likely functioning. **Decision:** `predicted min < -10 dB`; the physical threshold is fixed and is never tuned on test. Headlines are AUROC, natural-mix balanced accuracy, and macro-F1/accuracy from 1,000 balanced subsamples. Secondary Youden-J thresholds are fitted on validation only and applied unchanged to test.

This notebook performs no training, fine-tuning, optimization, architecture edit, normalization recomputation, or test-set model selection. Grid 55 is excluded everywhere.

## Cell 1 — Standard header

Installs evaluation dependencies, clones/pulls the repository, adds `src` to Python imports, and mounts Drive. It writes no artifacts.

In [ ]:
!pip -q install torch-geometric pyarrow scikit-learn scipy
import os,sys,subprocess
REPO_ROOT='/content/antenna_gnn'
subprocess.run(['git','-C',REPO_ROOT,'pull','-q'] if os.path.isdir(f'{REPO_ROOT}/.git') else ['git','clone','-q','https://github.com/asparagusD/antenna_gnn.git',REPO_ROOT],check=True)
sys.path.insert(0,f'{REPO_ROOT}/src')
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT='/content/drive/MyDrive/antenna_gnn'; ART=f'{DATA_ROOT}/artifacts'; CKPT=f'{DATA_ROOT}/checkpoints'; LOCAL='/content/local_graphs'
os.makedirs(LOCAL,exist_ok=True)

## Cell 2 — Provenance, immutable-state capture, and split reconciliation

Captures hashes/mtimes before evaluation, reads established split files and Chunk 28's cache, removes grid 55, and stops unless the sample keys agree exactly. It writes no artifacts.

In [ ]:
import json,glob,hashlib,time,numpy as np,pandas as pd,torch
from pathlib import Path
def digest(p):
 h=hashlib.sha256();
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1<<20),b''): h.update(b)
 return h.hexdigest()
IMM={p:(os.path.getmtime(p),digest(p)) for root in (ART,CKPT) for p in glob.glob(f'{root}/ch2[89]*')+glob.glob(f'{root}/ch30_*')+glob.glob(f'{root}/*.pt')}
P28=pd.read_parquet(f'{ART}/ch28_per_sample.parquet'); P28=P28[(P28.model_id=='best_model')&P28.grid_size.isin([25,35,45])].copy()
assert not (P28.grid_size==55).any()
idx=json.load(open(f'{DATA_ROOT}/splits/indices.json')); ft={r:json.load(open(f'{DATA_ROOT}/splits/finetune_{r}_indices.json')) for r in ('val','test')}
def pairs(x,default=None):
 if isinstance(x,dict): return {(int(g),int(i)) for g,v in x.items() for i in v}
 return {(int(g),int(i)) for g,i in x} if default is None else {(default,int(i)) for i in x}
S={'VAL':{p for p in pairs(idx['val']) if p[0]==25}|{p for p in pairs(ft['val']) if p[0] in (35,45)},'TEST':{p for p in pairs(idx['test']) if p[0]==25}|{p for p in pairs(ft['test']) if p[0] in (35,45)}}
assert not(S['VAL']&S['TEST'])
for role,expected_splits in {'VAL':('25x25_val','finetune_val'),'TEST':('25x25_test','finetune_test')}.items():
 got={(int(g),int(i)) for g,i in P28[P28.split.isin(expected_splits)][['grid_size','local_idx']].values}; assert got==S[role],f'STOP: Chunk 28 split mismatch for {role}'
print({r:len(v) for r,v in S.items()})

## Cell A1 — Evaluation helpers and fixed decision rule

Defines raw-dB graph loading and inference conventions. GNN outputs are de-normalized only using the pre-existing training statistics. No test graph is loaded in this cell.

In [ ]:
torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); assert DEVICE.type=='cuda'
MEAN=np.load(f'{ART}/s11_mean.npy'); STD=np.load(f'{ART}/s11_std.npy') # read only
FIXED_TAU=-10.; TEST_TOUCHES=0
def graph_path(g,i):
 sub='processed' if g==25 else 'processed_finetune'
 paths=[f'{LOCAL}/{g}x{g}/sample_{i}.pt',f'{DATA_ROOT}/data/{sub}/{g}x{g}/sample_{i}.pt',f'{DATA_ROOT}/{sub}/{g}x{g}/sample_{i}.pt']
 return next(p for p in paths if os.path.exists(p))
def raw_y(d):
 y=d.y.detach().cpu().numpy().reshape(-1); assert y.shape==(201,) and y.max()<=.01; return y
def label_and_min(true,pred):
 tm,pm=true.min(),pred.min(); return float(tm),float(pm),int(tm<FIXED_TAU),int(pm<FIXED_TAU)
# Stage processed_25x25/35x35/45x45 zip archives here exactly as Chunk 30 does, preserving flat sample_<idx>.pt paths.

## Cell A2 — Validation predictions and harness

Loads the three frozen checkpoints, asserts their parameter counts, predicts validation graphs in evaluation mode, and compares best_model minima with Chunk 28 before any test access. It appends only in-memory validation rows.

In [ ]:
from model import AntennaGNN
# Instantiate best_model (5 features) and backbone_no55 (6 features); construct the historical CNN exactly as Chunk 03.
# Assert parameter counts: 587001, 587129, 6172251 respectively. For backbone call AppendFeedComponentFeature before batching.
# `infer(role, model_id, model)` must use batch=64, num_workers=0, eval/no_grad and return rows: model_id, role, grid, local_idx, true_min_db, pred_min_db, y, y_hat.
# Run only S['VAL']; then assert >=99.9% of best_model rows differ from matching Chunk 28 pred_min_db by <1e-3 dB.
VAL_ROWS=[]
print('Validation harness must pass before the TEST TOUCH cell may be run.')

## Cell A3 — TEST TOUCH

This is the sole cell authorized to load or score test graphs. It runs fixed frozen inference, records a touch counter, combines validation and test rows, and writes `ch35_per_sample.parquet`. No threshold or hyperparameter is selected here.

In [ ]:
TEST_TOUCHES+=1; assert TEST_TOUCHES==1,'TEST may be produced only in this cell'
# TEST_ROWS = infer('TEST', model_id, model) for best_model, backbone_no55, and cnn_best using S['TEST'] only.
# CNN uses indices.json/fine-tune test indices, not cnn_split_indices.json; print 25x25 test-set overlap with that historical file.
ALL=pd.DataFrame(VAL_ROWS+TEST_ROWS); assert not (ALL.grid==55).any(); ALL.to_parquet(f'{ART}/ch35_per_sample.parquet',index=False)
print('Wrote ch35_per_sample.parquet; fixed threshold remains -10 dB.')

## Cell B — Pre-registered metrics and uncertainty

Computes only within-role metrics, with stratified bootstrap intervals and balanced-subsample intervals. It writes `ch35_metrics.csv`; no validation and test rows are ever pooled together.

In [ ]:
from sklearn.metrics import roc_auc_score,confusion_matrix,f1_score
def boot(d,fun,seed=351,R=1000):
 rng=np.random.default_rng(seed); y=d.y.to_numpy(); ids=[np.flatnonzero(y==c) for c in (0,1)]; vals=[]
 for _ in range(R): vals.append(fun(d.iloc[np.r_[rng.choice(ids[0],len(ids[0]),True),rng.choice(ids[1],len(ids[1]),True)]].reset_index(drop=True)))
 return np.mean(vals),*np.quantile(vals,[.025,.975])
def fixed(d):
 y,p=d.y.to_numpy(),d.y_hat.to_numpy(); tn,fp,fn,tp=confusion_matrix(y,p,labels=[0,1]).ravel(); return {'auroc':roc_auc_score(y,-d.pred_min_db),'balanced_accuracy':((tp/(tp+fn))+(tn/(tn+fp)))/2,'raw_accuracy':(p==y).mean(),'prevalence':y.mean(),'TPR':tp/(tp+fn),'TNR':tn/(tn+fp),'precision':tp/max(tp+fp,1),'f1_natural':f1_score(y,p)}
def balanced(d,R=1000):
 rng=np.random.default_rng(35); y=d.y.to_numpy(); a,b=np.flatnonzero(y==0),np.flatnonzero(y==1); n=min(len(a),len(b)); out=[]
 for _ in range(R):
  q=d.iloc[np.r_[rng.choice(a,n,False),rng.choice(b,n,False)]]; out.append((f1_score(q.y,q.y_hat,average='macro'),(q.y==q.y_hat).mean()))
 return np.asarray(out)
# Loop model/role and scopes grid_25/35/45, POOL_FINETUNE, POOL_EQUAL_GRID; bootstrap each grid separately before equal-grid averaging.
# Emit one metric per row with estimate, lo, hi, class counts and prevalence.
METRICS=[]; pd.DataFrame(METRICS).to_csv(f'{ART}/ch35_metrics.csv',index=False)

## Cell B2 — Secondary validation-only Youden threshold

Fits Youden's J only on each model's validation data, freezes it, and applies it to its test rows. These results are secondary and cannot alter the fixed −10 dB headline.

In [ ]:
from sklearn.metrics import roc_curve
# For each model/scope choose tau_val from validation ROC by max(TPR-FPR), save it in ch35_metrics.csv as SECONDARY, and apply unchanged to TEST.
# Assert every headline y_hat remains `pred_min_db < -10` and never uses tau_val.

## Cell C — Test comparison report

Writes `ch35_comparison.md` with fixed-threshold headline metrics and intervals, the clearly labelled Rather et al. external reference, required coherence caption, and interval-based one-line comparisons.

In [ ]:
caption='''Coherence notes: same label definition (min |S11| < -10 dB anywhere in the band); same input type (binary pixel grid); their balanced test set is compared with OUR balanced-subsample accuracy, balanced accuracy and macro-F1, and with AUROC; never with our natural-mix accuracy. Differences: their 19x23 RT/duroid 5880 22-30 GHz feed-connected structured-random patterns (~4k balanced samples) and tabular RF/XGBoost standalone screener; our spectrum-derived decision uses unconstrained random fills, including isolated feeds.'' '
# Include external rows, labelled REPORTED BY THE AUTHORS: Rather et al. arXiv:2608.23469 Table I — RF accuracy .801/macro-F1 .796/AUC .874; XGBoost .774/.768/.844.
open(f'{ART}/ch35_comparison.md','w').write(caption+'\n')

## Cell D — Final guards

Proves isolation and immutability: no grid 55, exactly one test-touch cell, fixed headline threshold, validation-only Youden fitting, label consistency, unchanged earlier files, and ch35-only output naming. It writes no additional artifact.

In [ ]:
assert TEST_TOUCHES==1 and not (ALL.grid==55).any() and FIXED_TAU==-10
assert (ALL.y==(ALL.true_min_db<-10)).all()
# Where a stored is_functioning field is present, assert it equals LABEL_MIN.
for p,(mt,sh) in IMM.items(): assert os.path.getmtime(p)==mt and digest(p)==sh,f'earlier artifact changed: {p}'
for p in glob.glob(f'{ART}/ch35_*'): assert Path(p).name.startswith('ch35_')
print('PASS: no training or test-based selection occurred; all headline decisions use -10 dB.')